# TradingAgents — FPT full graph, evidence audit and verified report download

Notebook mở trực tiếp trong Google Colab trên nhánh thử nghiệm riêng `fix/report-provenance-audit-20261028`. Nó giữ cấu trúc graph tiêu chuẩn của TradingAgents: đủ 4 analyst (market, social/sentiment, news, fundamentals), sau đó bull/bear research, research manager, trader, risk analysts và portfolio manager.

- SSI: nguồn giá OHLCV và chỉ báo kỹ thuật.
- FRED: API có khóa cho dữ liệu theo vintage/lịch sử; nếu không có `FRED_API_KEY`, chỉ dùng CSV công khai cho phân tích hiện tại, ghi rõ chưa khóa vintage và không dùng cho backtest lịch sử.
- Vnstock Fundamental thử VCI trước rồi KBS khi VCI lỗi; cả ba báo cáo tài chính trong một bản tóm tắt phải đến từ cùng một nguồn. Nếu Vnstock không dùng được, router mới chuyển sang Yahoo Finance/FPT IR theo cấu hình. KBS không tự tính TTM từ các dòng tiền cho đến khi xác minh được kỳ báo cáo là quý riêng hay lũy kế. Nguồn FPT IR chỉ trả về doanh thu/PBT/PAT/EPS; không thay thế bảng cân đối và lưu chuyển tiền tệ.
- Gemini: model cho các agent, có fallback ngay trong mỗi lần gọi LLM, không chỉ ở bước preflight.
- Báo cáo có mục Data Quality & Evidence Audit, ghi rõ thiếu dữ liệu, thiếu nguồn dẫn chứng và xung đột quyết định.
- Ô kiểm thử offline chạy trước phần credentials/AI; chỉ chạy ô cuối để tải ZIP sẽ không gọi Gemini/SSI.

Không có lệnh mua/bán. Đặt `SSI_API_KEY`, `SSI_API_SECRET` và `GOOGLE_API_KEY` trong Colab Secrets, không đưa key vào code. `VNSTOCK_API_KEY` cũng nên được thêm vào Colab Secrets cho lượt chạy dữ liệu Vnstock thật; nếu chưa có, adapter sẽ ghi nguồn không khả dụng và router thử nguồn dự phòng thay vì bịa số liệu. `FRED_API_KEY` là tùy chọn cho phiên chạy hiện tại nhưng cần thiết để truy vấn có vintage an toàn khi chạy ngày lịch sử. Bản này không sửa main và dùng thư mục /content/TradingAgentsFresh tách biệt.


In [ ]:
# 1) Setup: isolated checkout; does not change your old /content/TradingAgents folder.
import os
import subprocess
import sys
from pathlib import Path

# If code was already imported in this Python process, git checkout alone cannot
# refresh those module objects. Require a normal runtime restart before graph execution.
_TA_MODULES_LOADED_BEFORE_SETUP = any(
    name == "tradingagents" or name.startswith("tradingagents.")
    for name in sys.modules
)

REPO_URL = "https://github.com/PhamHuuQan/TradingAgents.git"
CODE_BRANCH = "fix/report-provenance-audit-20261028"
REPO_DIR = Path("/content/TradingAgentsFresh")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", CODE_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    # Clear local tracked edits before moving the branch pointer; otherwise checkout
    # can fail or preserve a malformed local .py file instead of the committed version.
    _tracked_changes_before_refresh = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"],
        text=True,
    ).strip()
    if _tracked_changes_before_refresh:
        print("Discarding local edits to tracked source files in the isolated checkout.")
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "reset", "--hard", "HEAD"],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", CODE_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "-f", "-B", CODE_BRANCH, "FETCH_HEAD"],
        check=True,
    )

# Restore tracked files exactly to the selected commit. Runtime outputs/checkpoints are
# untracked and are preserved; this checkout is dedicated to the notebook's source code.
subprocess.run(
    ["git", "-C", str(REPO_DIR), "reset", "--hard", "HEAD"],
    check=True,
)

# Catch syntax errors across source and tests before pytest collection.
subprocess.run(
    [sys.executable, "-m", "compileall", "-q", "tradingagents", "tests"],
    cwd=str(REPO_DIR),
    check=True,
)
print("Python syntax check passed for tradingagents/ and tests/.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
    cwd=str(REPO_DIR),
    check=True,
)

# Vnstock is an optional current-date-only provider for Vietnamese statements.
# If this optional install fails, the graph still runs and reports unavailable data.
try:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--extra-index-url", "https://vnstocks.com/api/packages", "vnstock",
        ],
        check=True,
    )
    print("Vnstock optional package installed.")
except subprocess.CalledProcessError:
    print("WARNING: Vnstock install failed; fundamentals tools will report unavailable data rather than invent values.")
# Always prioritize this isolated checkout, even if it already exists later in sys.path.
# Merely checking "if REPO_DIR not in sys.path" can leave an older /content/TradingAgents first.
import importlib
_REPO_RESOLVED = REPO_DIR.resolve()
sys.path[:] = [
    entry for entry in sys.path
    if str(Path(entry or ".").resolve()) != str(_REPO_RESOLVED)
]
sys.path.insert(0, str(_REPO_RESOLVED))
os.chdir(_REPO_RESOLVED)

# A git checkout does not refresh Python objects already cached in sys.modules.
# Evict them here, before the offline tests / SSI smoke check can import them again.
_TA_CACHED_MODULES = [
    name for name in list(sys.modules)
    if name == "tradingagents" or name.startswith("tradingagents.")
]
for _ta_name in sorted(_TA_CACHED_MODULES, key=lambda name: name.count("."), reverse=True):
    sys.modules.pop(_ta_name, None)
importlib.invalidate_caches()

# Discard notebook-level references to stale classes/functions from earlier cell runs.
for _ta_name in (
    "tradingagents", "ssi_smoke_main", "set_config", "get_vendor",
    "get_stock_data", "get_indicators", "GoogleClient", "_agent_tools",
    "_original_google_get_llm", "_original_route_to_vendor", "_safe_news_route",
    "DEFAULT_CONFIG", "TradingAgentsGraph", "ta", "final_state", "signal",
    "report_path", "report_dir", "price_vendor", "indicator_vendor",
    "price_result", "indicator_result", "GeminiFallbackRunnable",
):
    globals().pop(_ta_name, None)

# Fail early if Python would still import the package from another checkout.
_ta_spec = importlib.util.find_spec("tradingagents")
if (
    _ta_spec is None or _ta_spec.origin is None
    or not str(Path(_ta_spec.origin).resolve()).startswith(str(_REPO_RESOLVED))
):
    raise RuntimeError(
        "TradingAgents package resolution does not point to this isolated checkout. "
        f"Expected under {_REPO_RESOLVED}; found {getattr(_ta_spec, 'origin', None)}."
    )

_TA_CHECKED_OUT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
globals()["_TA_RESTART_REQUIRED"] = False

print(f"Repository ready: {CODE_BRANCH} @ {_TA_CHECKED_OUT_COMMIT}")
print(f"Import target verified: {_ta_spec.origin}")
if _TA_MODULES_LOADED_BEFORE_SETUP:
    print(
        f"Refreshed {len(_TA_CACHED_MODULES)} cached TradingAgents modules. "
        "No runtime restart is required for this stale-module condition."
    )
else:
    print("No cached TradingAgents modules needed eviction.")
print("Setup did not call Gemini or SSI.")


In [ ]:
# 2) Offline regression suite — no API keys, no Gemini calls, no SSI calls.
# Run this first. If it fails, isolate failing test files automatically.
import subprocess
import sys
from pathlib import Path

offline_tests = [
    "tests/test_report_quality.py",
    "tests/test_market_provenance.py",
    "tests/test_vietnam_market_ssi_vendor.py",
    "tests/test_sentiment_provenance.py",
    "tests/test_news_provenance.py",
    "tests/test_fpt_ir.py",
    "tests/test_fundamentals_units.py",
    "tests/test_vietnam_market_vnstock_vendor.py",
    "tests/test_vietnam_market_vnstock.py",
    "tests/test_news_lookahead.py",
    "tests/test_yahoo_snapshot.py",
    "tests/test_fred.py",
    "tests/test_polymarket.py",
    "tests/test_reddit_fallback.py",
    "tests/test_stocktwits_resilience.py",
    "tests/test_report_html.py",
    "tests/test_reporting.py",
]

pytest_log = Path("/content/evidence_quality_pytest.log")
cmd = [sys.executable, "-m", "pytest", "-q", "-ra", "--tb=short", *offline_tests]
print("Running offline evidence/provenance/report regression tests...", flush=True)
print("Command:", " ".join(cmd), flush=True)

result = subprocess.run(
    cmd, cwd=str(REPO_DIR), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
log_parts = ["===== COMPLETE OFFLINE TEST SUITE =====", result.stdout or "(pytest produced no output)"]
print(log_parts[-1], flush=True)

if result.returncode != 0:
    print("\n❌ Full suite failed; isolating each test file (still offline)...", flush=True)
    failures = []
    for test_file in offline_tests:
        single_cmd = [sys.executable, "-m", "pytest", "-q", "-ra", "--tb=short", test_file]
        single = subprocess.run(
            single_cmd, cwd=str(REPO_DIR), text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        status = "PASS" if single.returncode == 0 else "FAIL"
        print(f"[{status}] {test_file}", flush=True)
        if single.returncode != 0:
            failures.append(test_file)
            log_parts.extend([f"===== ISOLATED FAILURE: {test_file} =====", single.stdout or "(no output)"])
    log_parts.append("===== ISOLATION SUMMARY =====\n" + ("\n".join(failures) if failures else
        "Every file passes alone; the failure likely depends on test order/shared process state."))
    pytest_log.write_text("\n\n".join(log_parts), encoding="utf-8")
    print("\nFailing files:", failures if failures else "none when run independently", flush=True)
    print(f"Full suite and isolated failure logs saved to: {pytest_log}", flush=True)
    raise RuntimeError(
        f"Offline pytest failed (exit {result.returncode}). Failing test files: "
        f"{', '.join(failures) if failures else 'none in isolation (likely test interaction)'}. "
        f"Read {pytest_log}; do not run the SSI/Gemini analysis cell."
    )

pytest_log.write_text("\n\n".join(log_parts), encoding="utf-8")
print(f"Full pytest output saved to: {pytest_log}", flush=True)
print("✅ OFFLINE EVIDENCE TESTS PASSED. No Gemini/SSI API calls were made.", flush=True)


In [ ]:
# 3) Credentials and read-only SSI integration check. No order/trading endpoint is called.
from datetime import datetime, timedelta
from getpass import getpass
from zoneinfo import ZoneInfo

def read_colab_secret(name):
    current = os.environ.get(name)
    if isinstance(current, str) and current.strip():
        return current.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

def get_secret(name):
    return read_colab_secret(name) or getpass(f"{name} (input hidden): ")

ssi_key = get_secret("SSI_API_KEY")
ssi_secret = get_secret("SSI_API_SECRET")
if not ssi_key or not ssi_secret:
    raise ValueError("SSI_API_KEY and SSI_API_SECRET are required; values are never printed.")
os.environ["SSI_API_KEY"] = ssi_key
os.environ["SSI_API_SECRET"] = ssi_secret
del ssi_key, ssi_secret

# Optional Vnstock key: required by the live Vnstock package for automated notebook runs.
# Without it, the Vnstock adapter can be unavailable and the configured vendor router
# will continue to its next explicitly configured source; no values are fabricated.
vnstock_key = read_colab_secret("VNSTOCK_API_KEY")
if vnstock_key:
    os.environ["VNSTOCK_API_KEY"] = vnstock_key
    print("VNSTOCK_API_KEY available: Vnstock VCI → KBS fundamental fallback enabled.")
else:
    os.environ.pop("VNSTOCK_API_KEY", None)
    print("VNSTOCK_API_KEY not set: Vnstock may be unavailable; configured fallback vendors remain enabled.")
del vnstock_key

# Optional FRED key: use the vintage-pinned API for safer historical/as-of queries.
# Without it, the vendor permits only a keyless live/current CSV fallback.
fred_key = read_colab_secret("FRED_API_KEY")
if fred_key:
    os.environ["FRED_API_KEY"] = fred_key
    print("FRED_API_KEY available: vintage-pinned API enabled.")
else:
    os.environ.pop("FRED_API_KEY", None)
    print(
        "FRED_API_KEY not set: current/live CSV fallback only; historical/as-of "
        "FRED queries will remain unavailable rather than use unpinned revisions."
    )
del fred_key

VN_TODAY = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
FROM_DATE = (VN_TODAY - timedelta(days=30)).isoformat()
TO_DATE = VN_TODAY.isoformat()

from tradingagents.dataflows.vietnam_market.smoke import main as ssi_smoke_main
from tradingagents.dataflows.config import set_config
from tradingagents.dataflows.router import get_vendor
from tradingagents.agents.tools import get_stock_data, get_indicators

print(f"[1/2] SSI REST read-only check: FPT, {FROM_DATE} to {TO_DATE}")
exit_code = ssi_smoke_main([
    "--symbol", "FPT", "--from-date", FROM_DATE, "--to-date", TO_DATE, "--page-size", "100"
])
if exit_code != 0:
    raise RuntimeError("SSI REST check failed. Use the sanitized status message above; do not share credentials.")

set_config({"data_vendors": {"core_stock_apis": "ssi", "technical_indicators": "ssi"}})
price_vendor = get_vendor("core_stock_apis", "get_stock_data")
indicator_vendor = get_vendor("technical_indicators", "get_indicators")
print(f"[2/2] Router: prices={price_vendor}; indicators={indicator_vendor}")
if price_vendor.split(",")[0].strip() != "ssi" or indicator_vendor.split(",")[0].strip() != "ssi":
    raise RuntimeError("Router did not select SSI for both prices and indicators.")

price_result = get_stock_data.func("FPT", FROM_DATE, TO_DATE, TO_DATE)
if not isinstance(price_result, str) or "SSI FastConnect V3 read-only OHLCV for FPT" not in price_result:
    raise RuntimeError("Agent price tool did not return an SSI-sourced result.")
price_rows = [
    line for line in price_result.splitlines()
    if line.strip() and not line.startswith(("#", "Date,"))
]
if not price_rows:
    raise RuntimeError("SSI price tool returned no OHLCV rows.")

indicator_result = get_indicators.func("FPT", "rsi", TO_DATE, 30, TO_DATE)
if not isinstance(indicator_result, str) or "SSI FastConnect V3 indicator rsi for FPT" not in indicator_result:
    raise RuntimeError("Agent indicator tool did not return an SSI-sourced result.")
indicator_rows = [
    line for line in indicator_result.splitlines()
    if line.strip() and not line.startswith(("#", "Date,"))
]
if not indicator_rows:
    raise RuntimeError("SSI RSI tool returned no points.")

print(f"SSI + router + agent tools passed: {len(price_rows)} OHLCV rows; {len(indicator_rows)} RSI points.")
print("No credentials were printed or written into notebook cells.")


## 4) Lượt chạy AI và phục hồi khi lỗi

Bản này thêm kiểm toán chất lượng bằng chứng vào báo cáo, không chỉ cố ép graph chạy thành công:

- Nguồn giá/technical vẫn là SSI read-only.
- Nguồn cơ bản theo thứ tự `vnstock,yfinance,fpt_ir`: bên trong Vnstock, thử VCI trước rồi KBS; nếu cả hai không khả dụng mới chuyển sang Yahoo, sau đó mới xét FPT IR. Adapter FPT IR chỉ đọc bản công bố kết quả 9M mới nhất cho mã FPT, bắt buộc có URL/ngày công bố và khớp mẫu chỉ tiêu. Nguồn FPT IR không thay thế bảng cân đối/lưu chuyển tiền tệ; khi không có dữ liệu, báo cáo phải nêu rõ giới hạn.
- Các agent Analyst, Bull/Bear, Research Manager, Trader và Portfolio Manager nhận guardrail về bằng chứng, nguồn và dữ liệu thiếu.
- Cuối báo cáo có mục `VI. Data Quality & Evidence Audit` nêu trạng thái nguồn, thiếu URL/ngày, claim tài chính không có bằng chứng, thiếu mức thực thi hoặc hướng quyết định mâu thuẫn.
,- Bài test offline chạy ở ô 2, trước phần credentials; có thể chỉ chạy ô 1 rồi ô 2 để kiểm tra code mà không gọi SSI/Gemini. Ô phân tích thật mới tiêu thụ hạn mức Gemini; ô cuối chỉ đóng gói/tải ZIP.
,- Khi toàn bộ model trả 503/429, retry có giới hạn và checkpoint giúp tiếp tục node đã lưu trong cùng runtime; không đảm bảo khắc phục được khi quota của project đã cạn.


In [ ]:
# 5) OFFLINE GEMINI FALLBACK SELF-TEST — no credentials/network/API calls.
# This must pass before the next cell is allowed to make any model request.
import re
import time
from langchain_core.runnables import Runnable

def is_transient_gemini_error(exc):
    value = (type(exc).__name__ + " " + str(exc)).upper()
    return any(token in value for token in (
        "503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED",
        "500", "502", "504", "INTERNAL", "DEADLINE_EXCEEDED",
        "SERVICE UNAVAILABLE", "TIMED OUT", "SERVERERROR",
    ))

def _gemini_status(exc):
    for attr in ("status_code", "code"):
        value = getattr(exc, attr, None)
        if value is not None:
            return str(value)
    match = re.search(r"\b(408|429|500|502|503|504)\b", str(exc))
    return match.group(1) if match else type(exc).__name__

def is_daily_quota_error(exc):
    value = (type(exc).__name__ + " " + str(exc)).upper()
    return any(token in value for token in (
        "DAILY QUOTA", "QUOTA_EXCEEDED", "QUOTA EXCEEDED",
        "REQUESTS PER DAY", "TOKENS PER DAY", "RPD", "PER-DAY QUOTA",
    ))

class GeminiFallbackRunnable(Runnable):
    """Switch providers on transient failures; bounded exponential retry between full sweeps."""
    MAX_CYCLES = 3
    BACKOFF_SECONDS = (5, 20)

    def __init__(self, models, model_ids, shared_state, sleep_fn=None):
        super().__init__()
        if not models or len(models) != len(model_ids):
            raise ValueError("Gemini fallback model list is empty or inconsistent.")
        self.models = list(models)
        self.model_ids = list(model_ids)
        self.shared_state = shared_state
        self.sleep_fn = sleep_fn or time.sleep

    def invoke(self, input, config=None, **kwargs):
        if self.shared_state.get("exhausted"):
            last_model = self.shared_state.get("last_model", "unknown")
            last_status = self.shared_state.get("last_status", "unknown")
            raise RuntimeError(
                f"Gemini fallback exhausted for this graph step; "
                f"last_model={last_model}, status={last_status}. "
                "No additional model requests were sent."
            )

        quota_exhausted = set(self.shared_state.get("quota_exhausted_models", []))
        start = int(self.shared_state.get("active_index", 0))
        start = max(0, min(start, len(self.models) - 1))
        last_exc = None
        last_index = start

        for cycle in range(self.MAX_CYCLES):
            cycle_start = start if cycle == 0 else 0
            attempted_this_pass = 0
            for offset in range(len(self.models)):
                index = (cycle_start + offset) % len(self.models)
                model_id = self.model_ids[index]
                if model_id in quota_exhausted:
                    continue
                attempted_this_pass += 1
                try:
                    output = self.models[index].invoke(input, config=config, **kwargs)
                    self.shared_state.update({
                        "active_index": index,
                        "exhausted": False,
                        "last_model": model_id,
                        "last_status": None,
                    })
                    return output
                except Exception as exc:
                    if not is_transient_gemini_error(exc):
                        raise
                    last_exc = exc
                    last_index = index

                    if is_daily_quota_error(exc):
                        quota_exhausted.add(model_id)
                        self.shared_state.update({
                            "quota_exhausted_models": sorted(quota_exhausted),
                            "active_index": (index + 1) % len(self.models),
                            "last_model": model_id,
                            "last_status": _gemini_status(exc),
                        })
                        remaining_ids = [
                            candidate for candidate in self.model_ids
                            if candidate not in quota_exhausted
                        ]
                        if not remaining_ids:
                            self.shared_state["exhausted"] = True
                            raise RuntimeError(
                                "Daily quota/rate limit was reached for every configured Gemini model; "
                                f"models={', '.join(self.model_ids)}. No more requests will be sent. "
                                "Check AI Studio -> Usage and rate limits, or rerun after quota reset."
                            ) from None
                        next_index = (index + 1) % len(self.models)
                        print(
                            f"Daily quota appears exhausted for Gemini model {model_id}; "
                            f"skipping it and trying {self.model_ids[next_index]} if available."
                        )
                        continue

                    next_index = (index + 1) % len(self.models)
                    self.shared_state["active_index"] = next_index
                    print(
                        f"Gemini transient error on {model_id} "
                        f"(status={_gemini_status(exc)}); trying {self.model_ids[next_index]}."
                    )

            if attempted_this_pass == 0:
                self.shared_state.update({
                    "exhausted": True,
                    "last_model": self.shared_state.get("last_model", self.model_ids[last_index]),
                    "last_status": self.shared_state.get("last_status", "quota exhausted"),
                })
                raise RuntimeError(
                    "All configured Gemini model quotas are marked unavailable; "
                    "no additional model requests were sent."
                ) from None

            if cycle < self.MAX_CYCLES - 1:
                delay = self.BACKOFF_SECONDS[cycle]
                print(
                    f"All available Gemini models failed this pass; "
                    f"waiting {delay}s before retry pass {cycle + 2}/{self.MAX_CYCLES}."
                )
                self.shared_state["active_index"] = 0
                self.sleep_fn(delay)

        self.shared_state.update({
            "exhausted": True,
            "last_model": self.model_ids[last_index],
            "last_status": _gemini_status(last_exc) if last_exc else "unknown",
            "active_index": 0,
        })
        raise RuntimeError(
            f"All available Gemini fallback models failed after {self.MAX_CYCLES} passes; "
            f"last_model={self.model_ids[last_index]}, "
            f"status={self.shared_state['last_status']}. "
            "The graph checkpoint is retained; rerun the analysis cell to resume "
            "without replaying completed graph nodes."
        ) from None

    def bind_tools(self, tools, **kwargs):
        bound = [model.bind_tools(tools, **kwargs) for model in self.models]
        return GeminiFallbackRunnable(bound, self.model_ids, self.shared_state, self.sleep_fn)

    def with_structured_output(self, schema, **kwargs):
        bound = [model.with_structured_output(schema, **kwargs) for model in self.models]
        return GeminiFallbackRunnable(bound, self.model_ids, self.shared_state, self.sleep_fn)

class _FakeGemini:
    def __init__(self, name, failures=()):
        self.name = name
        self.failures = list(failures)
        self.calls = 0
    def invoke(self, input, config=None, **kwargs):
        self.calls += 1
        if self.failures:
            raise RuntimeError(self.failures.pop(0))
        return f"OK:{self.name}"
    def bind_tools(self, tools, **kwargs):
        return self
    def with_structured_output(self, schema, **kwargs):
        return self

# 1) Tool-calling invocation falls back on 503.
state = {"active_index": 0}
primary = _FakeGemini("primary", ["503 UNAVAILABLE"])
backup = _FakeGemini("backup")
chain = GeminiFallbackRunnable([primary, backup], ["primary", "backup"], state)
assert chain.bind_tools([]).invoke("test") == "OK:backup"
assert (primary.calls, backup.calls, state["active_index"]) == (1, 1, 1)

# 2) Structured output has the same fallback behavior.
state2 = {"active_index": 0}
structured = GeminiFallbackRunnable(
    [_FakeGemini("one", ["503 UNAVAILABLE"]), _FakeGemini("two")],
    ["one", "two"],
    state2,
)
assert structured.with_structured_output(dict).invoke("test") == "OK:two"
assert state2["active_index"] == 1

# 3) Non-transient 400 must not be hidden by a model switch.
state3 = {"active_index": 0}
bad_request = _FakeGemini("primary", ["400 INVALID_ARGUMENT"])
unused = _FakeGemini("backup")
chain3 = GeminiFallbackRunnable([bad_request, unused], ["primary", "backup"], state3)
try:
    chain3.invoke("test")
except RuntimeError as exc:
    assert "400 INVALID_ARGUMENT" in str(exc)
else:
    raise AssertionError("A non-transient 400 should be raised, not silently switched.")
assert unused.calls == 0 and state3["active_index"] == 0

# 4) One full sweep may fail transiently, but a later sweep can recover.
state_retry = {"active_index": 0}
retry_chain = GeminiFallbackRunnable(
    [
        _FakeGemini("primary-retry", ["503 UNAVAILABLE"]),
        _FakeGemini("backup-retry", ["503 UNAVAILABLE"]),
    ],
    ["primary-retry", "backup-retry"],
    state_retry,
    sleep_fn=lambda _seconds: None,
)
assert retry_chain.invoke("test") == "OK:primary-retry"
assert state_retry["active_index"] == 0

# 5) If every model fails across every pass, fail clearly and stop further requests.
state4 = {"active_index": 0}
all_fail_chain = GeminiFallbackRunnable(
    [
        _FakeGemini("one", ["503 UNAVAILABLE"] * 3),
        _FakeGemini("two", ["503 UNAVAILABLE"] * 3),
    ],
    ["one", "two"],
    state4,
    sleep_fn=lambda _seconds: None,
)
try:
    all_fail_chain.invoke("test")
except RuntimeError as exc:
    assert "after 3 passes" in str(exc)
else:
    raise AssertionError("Exhausted fallback chain must fail loudly.")
calls_before = (all_fail_chain.models[0].calls, all_fail_chain.models[1].calls)
try:
    all_fail_chain.invoke("test")
except RuntimeError as exc:
    assert "No additional model requests were sent" in str(exc)
else:
    raise AssertionError("Exhausted chain must not send extra calls on structured-output retry.")
assert calls_before == (all_fail_chain.models[0].calls, all_fail_chain.models[1].calls)


# 6) A per-model daily quota must skip that model and try the next configured model.
state_quota = {"active_index": 0}
quota_primary = _FakeGemini("quota-primary", ["429 RESOURCE_EXHAUSTED requests per day RPD"])
quota_backup = _FakeGemini("quota-backup")
quota_chain = GeminiFallbackRunnable(
    [quota_primary, quota_backup], ["quota-primary", "quota-backup"], state_quota
)
assert quota_chain.invoke("test") == "OK:quota-backup"
assert quota_primary.calls == 1 and quota_backup.calls == 1
assert state_quota["quota_exhausted_models"] == ["quota-primary"]
assert state_quota["exhausted"] is False

# 7) If every configured model has a daily quota error, do not repeat requests.
state_all_quota = {"active_index": 0}
quota_one = _FakeGemini("quota-one", ["429 RESOURCE_EXHAUSTED requests per day RPD"])
quota_two = _FakeGemini("quota-two", ["429 RESOURCE_EXHAUSTED requests per day RPD"])
all_quota_chain = GeminiFallbackRunnable(
    [quota_one, quota_two], ["quota-one", "quota-two"], state_all_quota
)
try:
    all_quota_chain.invoke("test")
except RuntimeError as exc:
    assert "every configured Gemini model" in str(exc)
else:
    raise AssertionError("All-model daily quota exhaustion must fail clearly.")
quota_calls_before = (quota_one.calls, quota_two.calls)
try:
    all_quota_chain.invoke("test")
except RuntimeError as exc:
    assert "No additional model requests were sent" in str(exc)
else:
    raise AssertionError("Exhausted quota chain must not send more requests.")
assert quota_calls_before == (quota_one.calls, quota_two.calls)

print("✅ OFFLINE SELF-TEST PASSED: 7/7 cases; no API key read and no Gemini/SSI API calls made.")


In [ ]:
# 6) One full TradingAgents graph run, with per-call Gemini fallback.

if globals().get("_TA_RESTART_REQUIRED"):
    # Last-resort repair if an earlier cell re-set the stale-module flag.
    # Setup now evicts old modules and verifies import paths before other notebook cells run.
    import importlib as _ta_importlib
    import sys as _ta_sys
    from pathlib import Path as _ta_Path

    _ta_cached_modules = [
        name for name in list(_ta_sys.modules)
        if name == "tradingagents" or name.startswith("tradingagents.")
    ]
    for _ta_name in sorted(_ta_cached_modules, key=lambda name: name.count("."), reverse=True):
        _ta_sys.modules.pop(_ta_name, None)
    _ta_importlib.invalidate_caches()

    _ta_repo_path = str(REPO_DIR.resolve())
    _ta_sys.path[:] = [
        entry for entry in _ta_sys.path
        if str(_ta_Path(entry or ".").resolve()) != _ta_repo_path
    ]
    _ta_sys.path.insert(0, _ta_repo_path)

    for _ta_name in (
        "GoogleClient", "_agent_tools", "_original_google_get_llm",
        "_original_route_to_vendor", "_safe_news_route",
    ):
        globals().pop(_ta_name, None)

    globals()["_TA_RESTART_REQUIRED"] = False
    print(
        f"Refreshed {len(_ta_cached_modules)} cached TradingAgents modules from "
        f"{REPO_DIR}; graph imports will use the isolated checkout.",
        flush=True,
    )

from copy import deepcopy
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import os
import re
import time

google_key = get_secret("GOOGLE_API_KEY")
if not google_key:
    raise ValueError("GOOGLE_API_KEY is empty.")
os.environ["GOOGLE_API_KEY"] = google_key
del google_key

# These stable model IDs are listed in Google's current Gemini API catalog.
MODEL_CANDIDATES = [
    # Only model IDs exposed by this repository's Google model catalog.
    # 3.6 Flash is omitted here because this checkout warns that it is unknown.
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.8-flash",
]

from langchain_google_genai import ChatGoogleGenerativeAI

def safe_gemini_error(exc):
    message = str(exc)
    key = os.environ.get("GOOGLE_API_KEY", "")
    if key:
        message = message.replace(key, "[REDACTED]")
    return message[:500]

def is_transient_gemini_error(exc):
    value = (type(exc).__name__ + " " + str(exc)).upper()
    return any(token in value for token in (
        "503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED",
        "500", "502", "504", "INTERNAL", "DEADLINE_EXCEEDED",
        "SERVICE UNAVAILABLE", "TIMED OUT", "SERVERERROR",
    ))

def _gemini_status(exc):
    for attr in ("status_code", "code"):
        value = getattr(exc, attr, None)
        if value is not None:
            return str(value)
    match = re.search(r"\b(408|429|500|502|503|504)\b", str(exc))
    return match.group(1) if match else type(exc).__name__

# Keep the selected initial model stable if this cell is rerun after a checkpointed failure.
# This avoids an extra Gemini preflight request and keeps the checkpoint signature identical.
prior_model = globals().get("_TA_INITIAL_MODEL_ID") or globals().get("MODEL_ID")
if prior_model in MODEL_CANDIDATES:
    MODEL_ID = prior_model
    globals()["_TA_INITIAL_MODEL_ID"] = MODEL_ID
    print(f"Reusing model from this runtime for checkpoint continuity: {MODEL_ID}")
else:
    MODEL_ID = None
    preflight_errors = []
    for candidate in MODEL_CANDIDATES:
        try:
            ping = ChatGoogleGenerativeAI(
                model=candidate,
                google_api_key=os.environ["GOOGLE_API_KEY"],
                max_retries=0,
                max_output_tokens=24,
                timeout=45,
            ).invoke("Reply with exactly the single word READY.")
            if not getattr(ping, "content", None):
                raise RuntimeError("Gemini returned an empty response.")
            MODEL_ID = candidate
            print(f"Gemini preflight passed: {MODEL_ID}")
            break
        except Exception as exc:
            preflight_errors.append(f"{candidate}: {type(exc).__name__}, status={_gemini_status(exc)}")
            upper = (type(exc).__name__ + " " + str(exc)).upper()
            if any(token in upper for token in (
                "401", "403", "API_KEY_INVALID", "PERMISSION_DENIED",
                "API KEY NOT VALID", "BILLING",
            )):
                raise RuntimeError(
                    f"Gemini authentication/project access failed ({type(exc).__name__}): {safe_gemini_error(exc)}"
                ) from None
            print(f"Gemini preflight failed for {candidate} (status={_gemini_status(exc)}); trying next model.")

    if MODEL_ID is None:
        raise RuntimeError(
            "No configured Gemini model passed preflight. Graph was not started.\n"
            + "\n".join(preflight_errors)
        )
    globals()["_TA_INITIAL_MODEL_ID"] = MODEL_ID

from tradingagents.llm_clients.google_client import GoogleClient

# Install fallback at the provider-client boundary. Original graph agents/prompts/tools
# remain intact; only a transient Gemini transport/rate-limit failure is rerouted.
# Idempotent monkeypatch: rerunning this cell must not wrap the wrapper recursively.
if not hasattr(GoogleClient, "_ta_original_get_llm"):
    _preexisting_original = globals().get("_original_google_get_llm")
    if callable(_preexisting_original) and getattr(_preexisting_original, "__name__", "") == "get_llm":
        GoogleClient._ta_original_get_llm = _preexisting_original
    else:
        GoogleClient._ta_original_get_llm = GoogleClient.get_llm
_original_google_get_llm = GoogleClient._ta_original_get_llm
# Preserve model-specific quota exclusions across an interrupted checkpoint resume.
_previous_runtime_state = globals().get("_runtime_state")
if isinstance(_previous_runtime_state, dict):
    _runtime_state = dict(_previous_runtime_state)
else:
    _runtime_state = {}
_runtime_state["exhausted"] = False
_runtime_state.setdefault("active_index", 0)
_runtime_state.setdefault("quota_exhausted_models", [])
RUNTIME_MODEL_IDS = [MODEL_ID] + [model for model in MODEL_CANDIDATES if model != MODEL_ID]

def _google_get_llm_with_runtime_fallback(self):
    ids = [self.model] + [model for model in MODEL_CANDIDATES if model != self.model]
    instances = [
        _original_google_get_llm(GoogleClient(model_id, self.base_url, **self.kwargs))
        for model_id in ids
    ]
    return GeminiFallbackRunnable(instances, ids, _runtime_state)

GoogleClient.get_llm = _google_get_llm_with_runtime_fallback

# For FPT, overseas news/insider sources may not cover the ticker. A failed
# enrichment returns an explicit sentinel instead of aborting the entire graph.
from tradingagents.agents import tools as _agent_tools
if not hasattr(_agent_tools, "_ta_original_route_to_vendor"):
    _preexisting_route = globals().get("_original_route_to_vendor")
    if callable(_preexisting_route) and getattr(_preexisting_route, "__name__", "") == "route_to_vendor":
        _agent_tools._ta_original_route_to_vendor = _preexisting_route
    else:
        _agent_tools._ta_original_route_to_vendor = _agent_tools.route_to_vendor
_original_route_to_vendor = _agent_tools._ta_original_route_to_vendor
_NEWS_METHODS = {"get_news", "get_global_news", "get_insider_transactions"}

def _safe_news_route(method, *args, **kwargs):
    try:
        return _original_route_to_vendor(method, *args, **kwargs)
    except Exception as exc:
        if method in _NEWS_METHODS:
            print(f"News enrichment unavailable for {method}: {type(exc).__name__}; report as unavailable.")
            return (
                f"DATA_UNAVAILABLE: {method} could not be retrieved from the configured news provider "
                f"({type(exc).__name__}). Do not infer or fabricate news/insider events."
            )
        raise

_agent_tools.route_to_vendor = _safe_news_route

from tradingagents.default_config import DEFAULT_CONFIG
from tradingagents.graph.trading_graph import TradingAgentsGraph
import tradingagents.report_quality as _ta_report_quality_module
import tradingagents.reporting as _ta_reporting_module

# Record and enforce the exact audit/report modules used by this graph. This catches
# notebook/import-path drift before spending Gemini calls on a report that uses stale audit code.
_expected_root = REPO_DIR.resolve()
_loaded_report_modules = {
    "report_quality": Path(_ta_report_quality_module.__file__).resolve(),
    "reporting": Path(_ta_reporting_module.__file__).resolve(),
}
for _module_name, _module_path in _loaded_report_modules.items():
    try:
        _module_path.relative_to(_expected_root)
    except ValueError:
        raise RuntimeError(
            f"Stale/wrong {_module_name} module loaded from {_module_path}; "
            f"expected it under {_expected_root}. Rerun the latest notebook Setup cell."
        ) from None
print(f"Verified checkout commit: {globals().get('_TA_CHECKED_OUT_COMMIT', 'unknown')}")
for _module_name, _module_path in _loaded_report_modules.items():
    print(f"Verified {_module_name} module: {_module_path}")

config = deepcopy(DEFAULT_CONFIG)
# Store the exact checkout revision in every generated report so output can be tied
# to the code which actually produced it. The SHA also isolates checkpoints across code changes.
config["code_branch"] = CODE_BRANCH
config["code_commit"] = str(globals().get("_TA_CHECKED_OUT_COMMIT") or subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip())
config["llm_provider"] = "google"
config["quick_think_provider"] = None
config["deep_think_provider"] = None
config["quick_think_llm"] = MODEL_ID
config["deep_think_llm"] = MODEL_ID
config["output_language"] = "Vietnamese"
# Use the runtime fallback immediately instead of spending retries on one overloaded model.
config["llm_max_retries"] = 0
# Enable per-node SQLite checkpoints. After a transient LLM failure, rerunning this cell
# with the same date/model/settings resumes from the last successful graph checkpoint.
config["checkpoint_enabled"] = True
config["data_vendors"] = dict(config.get("data_vendors", {}))
config["data_vendors"]["core_stock_apis"] = "ssi"
config["data_vendors"]["technical_indicators"] = "ssi"
# Vnstock VCI may deny statement endpoints on a community entitlement; try Yahoo next.
# If neither source has verified figures, agents must report them unavailable.
config["data_vendors"]["fundamental_data"] = "vnstock,yfinance,fpt_ir"
config["results_dir"] = str(REPO_DIR / "results")
config["data_cache_dir"] = str(REPO_DIR / ".runtime" / "cache")
config["memory_log_path"] = str(REPO_DIR / ".runtime" / "trading_memory.md")
Path(config["results_dir"]).mkdir(parents=True, exist_ok=True)
Path(config["data_cache_dir"]).mkdir(parents=True, exist_ok=True)
Path(config["memory_log_path"]).parent.mkdir(parents=True, exist_ok=True)

# Keep the trade date stable when resuming a checkpoint in this same runtime.
ANALYSIS_DATE = globals().get("_TA_ANALYSIS_DATE") or datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date().isoformat()
globals()["_TA_ANALYSIS_DATE"] = ANALYSIS_DATE
ANALYSTS = ["market", "social", "news", "fundamentals"]
print(f"Starting full TradingAgents graph: FPT / {ANALYSIS_DATE}; initial_model={MODEL_ID}; commit={config['code_commit']}")
print(f"Analysts: {', '.join(ANALYSTS)} -> research/debate -> trader -> risk -> portfolio manager")
print("Sources: SSI OHLCV/indicators; Vnstock -> Yahoo Finance -> official FPT IR 9M summary; statements remain unavailable unless a source returns them.")
print("Current-date source contract remains in force; this is not a historical point-in-time backtest.")
print("Checkpointing enabled: a later rerun can resume completed graph nodes if this run fails.")

ta = TradingAgentsGraph(selected_analysts=ANALYSTS, config=config, debug=False)
final_state, signal = ta.propagate("FPT", ANALYSIS_DATE, asset_type="stock")
report_path = Path(ta.save_reports(final_state, "FPT", html=True)).resolve()
if report_path.is_dir():
    report_dir = report_path
    report_path = report_dir / "complete_report.md"
else:
    report_dir = report_path.parent

if not report_path.is_file() or report_path.stat().st_size <= 0:
    raise RuntimeError(f"TradingAgents finished but complete_report.md is missing/empty: {report_path}")

print("Full TradingAgents graph completed; complete_report.md is non-empty.")
print(f"Active Gemini model at completion: {RUNTIME_MODEL_IDS[min(_runtime_state['active_index'], len(RUNTIME_MODEL_IDS) - 1)]}")
print(f"Signal/rating: {signal}")
print(f"Report size: {report_path.stat().st_size:,} bytes")
print(f"Report directory: {report_dir}")
print("Run only the final cell if you want to package/download this report again.")


In [ ]:
# 7) PACKAGE + DOWNLOAD EXISTING REPORT ONLY.
# This cell never calls Gemini or SSI. Run only this cell to download the report again.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files as colab_files

def find_existing_fpt_report(candidate=None):
    if candidate is not None:
        p = Path(candidate).expanduser()
        if p.is_file() and (p.name.lower() == "complete_report.md" or p.suffix.lower() == ".md"):
            return p.resolve()
        if p.is_dir():
            expected = p / "complete_report.md"
            if expected.is_file():
                return expected.resolve()

    roots = [
        Path("/root/.tradingagents/logs/reports"),
        Path("/content/TradingAgentsFresh/results/reports"),
        Path("/content/results/reports"),
    ]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        if root.is_file() and root.suffix.lower() == ".md":
            matches.append(root)
        elif root.is_dir():
            matches.extend(
                p for p in root.rglob("complete_report.md")
                if p.is_file() and p.parent.name.startswith("FPT_")
            )
    if not matches:
        raise FileNotFoundError(
            "No existing FPT complete_report.md found in this runtime. "
            "This cell will not rerun AI; the earlier runtime may have been reset."
        )
    return max(set(p.resolve() for p in matches), key=lambda p: p.stat().st_mtime)

candidate = globals().get("report_path")
if candidate is None:
    candidate = globals().get("report_dir")
existing_report = find_existing_fpt_report(candidate)

if existing_report.stat().st_size <= 0:
    raise RuntimeError(f"Report is empty; refusing to create ZIP: {existing_report}")

existing_dir = existing_report.parent
items = sorted(
    p for p in existing_dir.rglob("*")
    if p.is_file() and not p.is_symlink()
)
if existing_report not in items:
    items.append(existing_report)
items = sorted(set(items))
if not items:
    raise RuntimeError(f"No files found for packaging: {existing_dir}")

archive = Path("/content/TradingAgents_FPT_report.zip")
temp_archive = Path(str(archive) + ".tmp")
archive.parent.mkdir(parents=True, exist_ok=True)
temp_archive.unlink(missing_ok=True)

try:
    with ZipFile(temp_archive, "w", compression=ZIP_DEFLATED) as zf:
        for item in items:
            zf.write(item, arcname=f"{existing_dir.name}/{item.relative_to(existing_dir)}")

    with ZipFile(temp_archive, "r") as zf:
        names = zf.namelist()
        corrupt = zf.testzip()
        expected = f"{existing_dir.name}/{existing_report.relative_to(existing_dir)}"
        if not names or corrupt:
            raise RuntimeError(f"ZIP verification failed: empty={not names}; corrupt member={corrupt}")
        if expected not in names or zf.getinfo(expected).file_size <= 0:
            raise RuntimeError("ZIP verification failed: complete_report.md missing or empty")
    if temp_archive.stat().st_size <= 0:
        raise RuntimeError("ZIP archive has zero bytes.")
    temp_archive.replace(archive)
finally:
    temp_archive.unlink(missing_ok=True)

print(f"ZIP VERIFIED: {len(names)} files; {archive.stat().st_size:,} bytes.")
print(f"Report inside ZIP: {expected}; report size={existing_report.stat().st_size:,} bytes.")
for name in names:
    print(" -", name)
print("Starting browser download. This cell makes no Gemini/SSI calls.")
colab_files.download(str(archive))
